# Capstone 2 — House Price Estimation Using Regression (PropPulse Chennai)

In [ ]:
import numpy as np
import pandas as pd

def generate_chennai_housing_dataset(n_samples=2500, seed=42):
    np.random.seed(seed)

    localities = {
        "Anna Nagar": 11800,
        "Adyar": 13200,
        "Velachery": 8400,
        "Porur": 7100,
        "OMR Thoraipakkam": 7800,
        "Tambaram": 5200,
        "Chromepet": 6000,
        "Perambur": 6700,
    }
    area_names = list(localities.keys())
    area_probs = [0.12, 0.08, 0.16, 0.18, 0.18, 0.12, 0.08, 0.08]

    chosen_areas = np.random.choice(area_names, size=n_samples, p=area_probs)
    base_rates = np.array([localities[a] for a in chosen_areas])

    int_sqft = np.random.normal(loc=1250, scale=420, size=n_samples).clip(480, 4200).round(0)
    
    n_bedroom = np.where(int_sqft < 850, 1,
                np.where(int_sqft < 1450, 2,
                np.where(int_sqft < 2200, 3, 4))).astype(float)
    
    n_bathroom = np.clip(n_bedroom - np.random.choice([0, 1], size=n_samples, p=[0.75, 0.25]), 1, 4).astype(float)
    n_room = n_bedroom + n_bathroom + np.random.choice([1, 2], size=n_samples)

    property_age = np.random.exponential(scale=8.5, size=n_samples).clip(0, 35).round(1)
    dist_mainroad = np.random.uniform(10, 500, size=n_samples).round(0)

    buildtype = np.random.choice(["Residential", "Commercial", "Mixed"], size=n_samples, p=[0.74, 0.16, 0.10])
    park_facil = np.random.choice(["Yes", "No"], size=n_samples, p=[0.78, 0.22])
    sale_cond = np.random.choice(["Normal Sale", "Family", "AbNormal", "Partial", "AdjLand"], size=n_samples, p=[0.65, 0.10, 0.10, 0.08, 0.07])

    build_multiplier = np.where(buildtype == "Commercial", 1.28, np.where(buildtype == "Mixed", 1.12, 1.0))
    park_bonus = np.where(park_facil == "Yes", 4.5, 0.0)
    age_depreciation = np.maximum(0.68, 1.0 - (property_age * 0.011))
    road_discount = np.where(dist_mainroad > 300, -2.5, 1.5)

    raw_price_lakhs = (
        ((int_sqft * base_rates) / 100000.0) * build_multiplier * age_depreciation
        + (n_bedroom * 3.2) + park_bonus + road_discount
    )
    market_noise = np.random.normal(0, raw_price_lakhs * 0.085, size=n_samples)
    price_lakhs = np.clip(raw_price_lakhs + market_noise, 18.0, None).round(2)

    outlier_idx = np.random.choice(n_samples, size=int(n_samples * 0.015), replace=False)
    price_lakhs[outlier_idx] = price_lakhs[outlier_idx] * np.random.uniform(2.2, 3.5, size=len(outlier_idx))
    price_lakhs = price_lakhs.round(2)

    price_per_sqft = ((price_lakhs * 100000) / int_sqft).round(0)
    reg_fee_lakhs = (price_lakhs * 0.07 + np.random.normal(0, 0.15, size=n_samples)).clip(0.5, None).round(2)
    commis_lakhs = (price_lakhs * 0.015 + np.random.normal(0, 0.08, size=n_samples)).clip(0.1, None).round(2)

    df = pd.DataFrame({
        "PRT_ID": [f"CHN_{10000 + i}" for i in range(n_samples)],
        "AREA": chosen_areas,
        "INT_SQFT": int_sqft,
        "N_BEDROOM": n_bedroom,
        "N_BATHROOM": n_bathroom,
        "N_ROOM": n_room,
        "PROPERTY_AGE": property_age,
        "DIST_MAINROAD": dist_mainroad,
        "BUILDTYPE": buildtype,
        "PARK_FACIL": park_facil,
        "SALE_COND": sale_cond,
        "PRICE_PER_SQFT": price_per_sqft,
        "REG_FEE_LAKHS": reg_fee_lakhs,
        "COMMIS_LAKHS": commis_lakhs,
        "PRICE_LAKHS": price_lakhs
    })

    for col, frac in [("N_BEDROOM", 0.03), ("N_BATHROOM", 0.04), ("PARK_FACIL", 0.035), ("PROPERTY_AGE", 0.025)]:
        mask = np.random.choice(n_samples, size=int(n_samples * frac), replace=False)
        df.loc[mask, col] = np.nan

    df.to_csv("chennai_housing_raw.csv", index=False)
    return df

if __name__ == "__main__":
    generate_chennai_housing_dataset()


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from generate_chennai_data import generate_chennai_housing_dataset
df_raw = generate_chennai_housing_dataset()

LEAKAGE_COLS = ["PRT_ID", "PRICE_PER_SQFT", "REG_FEE_LAKHS", "COMMIS_LAKHS"]
df_clean = df_raw.drop(columns=LEAKAGE_COLS).copy()

upper_cap = df_clean["PRICE_LAKHS"].quantile(0.99)
df_clean = df_clean[df_clean["PRICE_LAKHS"] <= upper_cap].reset_index(drop=True)

# STEP 5: EDA VISUALISATIONS
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("PropPulse Chennai — Exploratory Data Analysis (Step 5)", fontsize=16, fontweight="bold")

sns.scatterplot(data=df_clean.sample(600, random_state=42), x="INT_SQFT", y="PRICE_LAKHS", hue="BUILDTYPE", alpha=0.75, ax=axes[0, 0])
axes[0, 0].set_title("1. Interior Area (SqFt) vs Price (₹ Lakhs)")

order_loc = df_clean.groupby("AREA")["PRICE_LAKHS"].median().sort_values(ascending=False).index
sns.boxplot(data=df_clean, x="AREA", y="PRICE_LAKHS", order=order_loc, ax=axes[0, 1], palette="Blues_r")
axes[0, 1].set_title("2. Price Distribution Across Chennai Corridors")
axes[0, 1].tick_params(axis="x", rotation=30)

sns.violinplot(data=df_clean, x="N_BEDROOM", y="PRICE_LAKHS", ax=axes[1, 0], palette="crest")
axes[1, 0].set_title("3. Price Distribution by Bedroom Count (BHK)")

sns.regplot(data=df_clean.sample(600, random_state=42), x="PROPERTY_AGE", y="PRICE_LAKHS",
            scatter_kws={"alpha": 0.3, "color": "#475569"}, line_kws={"color": "#dc2626", "linewidth": 2}, ax=axes[1, 1])
axes[1, 1].set_title("4. Property Age Depreciation Curve")

plt.tight_layout()
plt.savefig("eda_visualisations.png", dpi=200)
plt.close()

# STEP 4: REPRODUCIBLE PREPROCESSING PIPELINE
NUM_FEATURES = ["INT_SQFT", "N_BEDROOM", "N_BATHROOM", "N_ROOM", "PROPERTY_AGE", "DIST_MAINROAD"]
CAT_FEATURES = ["AREA", "BUILDTYPE", "PARK_FACIL", "SALE_COND"]
TARGET = "PRICE_LAKHS"

X = df_clean[NUM_FEATURES + CAT_FEATURES]
y = df_clean[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUM_FEATURES),
    ("cat", categorical_transformer, CAT_FEATURES)
])

models = {
    "0. Baseline (Median Dummy)": DummyRegressor(strategy="median"),
    "1. Ridge Linear Regression": Ridge(alpha=1.0),
    "2. Random Forest Regressor": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "3. Gradient Boosting (Champion)": GradientBoostingRegressor(loss="squared_error", n_estimators=250, learning_rate=0.06, max_depth=4, random_state=42)
}

results = []
trained_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results.append({
        "Model": name,
        "MAE (₹ Lakhs)": round(mean_absolute_error(y_test, preds), 2),
        "RMSE (₹ Lakhs)": round(np.sqrt(mean_squared_error(y_test, preds)), 2),
        "R² Score": round(r2_score(y_test, preds), 4)
    })
    trained_pipelines[name] = pipe

results_df = pd.DataFrame(results)

champion_pipe = trained_pipelines["3. Gradient Boosting (Champion)"]
lower_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", GradientBoostingRegressor(loss="quantile", alpha=0.05, n_estimators=250, learning_rate=0.06, max_depth=4, random_state=42))])
upper_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", GradientBoostingRegressor(loss="quantile", alpha=0.95, n_estimators=250, learning_rate=0.06, max_depth=4, random_state=42))])

lower_pipe.fit(X_train, y_train)
upper_pipe.fit(X_train, y_train)
y_pred_point = champion_pipe.predict(X_test)

perm_imp = permutation_importance(champion_pipe, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm_imp.importances_mean,
    "Importance_Std": perm_imp.importances_std
}).sort_values("Importance_Mean", ascending=True)

fig2, axes2 = plt.subplots(1, 2, figsize=(15, 6))
axes2[0].scatter(y_test.values, y_pred_point, alpha=0.5, color="#2563eb")
max_val = max(y_test.max(), y_pred_point.max())
axes2[0].plot([15, max_val], [15, max_val], "r--", lw=2)
axes2[0].set_title("Actual vs Predicted House Prices (₹ Lakhs)")
axes2[0].set_xlabel("Actual Price (₹ Lakhs)")
axes2[0].set_ylabel("Predicted Price (₹ Lakhs)")

axes2[1].barh(imp_df["Feature"], imp_df["Importance_Mean"], xerr=imp_df["Importance_Std"], color="#0d9488")
axes2[1].set_title("Permutation Importance (R² Drop on Shuffle)")
axes2[1].set_xlabel("Decrease in R² Score")

plt.tight_layout()
plt.savefig("model_diagnostics_and_xai.png", dpi=200)
plt.close()

bundle = {
    "point_model": champion_pipe,
    "lower_model": lower_pipe,
    "upper_model": upper_pipe,
    "metrics": results_df.to_dict(orient="records"),
    "feature_importance": imp_df.sort_values("Importance_Mean", ascending=False).to_dict(orient="records"),
    "localities": sorted(df_clean["AREA"].unique().tolist()),
    "buildtypes": sorted(df_clean["BUILDTYPE"].unique().tolist()),
}
joblib.dump(bundle, "chennai_price_bundle.pkl")
